# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_02_cnn_rnn/01_cnn_essentials.ipynb)

이 노트북은 "밑바닥부터 시작하는 딥러닝" 3장 (MLP, numpy 직접 구현) 까지 마친 학습자가
**CNN 의 본질**을 이해하기 위한 노트북이다.

흐름:

1. 합성곱 (cross-correlation) 의 수학적 정의
2. numpy 로 2D conv 직접 구현
3. padding, stride 와 출력 크기 공식
4. PyTorch `nn.Conv2d` 와 결과 일치 검증
5. 채널 (입력/출력 채널) 과 파라미터 수
6. Pooling (Max / Avg)
7. 작은 CNN 으로 실제 학습 (1-2 epoch)
8. CNN 의 inductive bias (locality / weight sharing) — MLP 와 비교
9. 흔한 함정 (모양 순서, 출력 크기 계산)
10. 이미 만들어진 큰 모델 (`torchvision.models`) 잠깐 맛보기

> 핵심 메시지: **CNN 은 "MLP 의 가중치 행렬에 강한 사전지식을 박아 넣은 것"** 이다.
> 그 사전지식이 (1) **locality** (가까운 픽셀끼리만 본다) 와
> (2) **weight sharing** (같은 필터를 모든 위치에 적용한다) 두 가지다.

In [ ]:
# 공통 실행 환경 준비 (Colab / Local 자동 분기)
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_02_cnn_rnn'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

# torch 가 없으면 설치 (Colab 에는 기본 설치되어 있음)
try:
    import torch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa: F401


In [ ]:
# 핵심 import 및 재현성 설정
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 20190501
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"torch version: {torch.__version__}")


# 1. 합성곱(Cross-Correlation) 의 정의

수학에서의 진짜 convolution 은 커널을 뒤집어서 (flip) 정의하지만,
**딥러닝에서 "convolution" 이라고 부르는 연산은 사실 cross-correlation** 이다 (kernel 을 flip 하지 않는다).
어차피 커널 자체를 학습으로 정하기 때문에 flip 여부는 결과 품질에 영향을 주지 않는다.

2D 입력 $I$ 와 2D 커널 $K$ 에 대한 cross-correlation 은:

$$ (I * K)(i, j) = \sum_{m}\sum_{n} I(i+m,\; j+n) \cdot K(m, n) $$

직관:
- 커널이 입력 위를 한 칸씩 미끄러진다 (sliding window).
- 각 위치에서 입력의 부분 patch 와 커널을 **원소별로 곱한 뒤 모두 더한다** (점곱).
- 결과 한 칸 = 점곱 한 번.

> MLP 와의 차이는 **"전체 입력에 행렬을 곱한다"** 가 아니라
> **"입력의 작은 patch 에만 작은 가중치를 곱한다"** 는 점이다.
> 그리고 그 patch 가 위치를 옮겨가며 **같은 커널** 을 반복 사용한다 — weight sharing.

In [ ]:
def conv2d_numpy(x: np.ndarray, k: np.ndarray) -> np.ndarray:
    """numpy 로 직접 구현한 2D cross-correlation (padding 0, stride 1).

    Args:
        x: shape (H, W) 입력
        k: shape (kh, kw) 커널

    Returns:
        shape (H - kh + 1, W - kw + 1) 출력
    """
    H, W = x.shape
    kh, kw = k.shape
    out_h = H - kh + 1
    out_w = W - kw + 1
    y = np.zeros((out_h, out_w), dtype=x.dtype)
    for i in range(out_h):
        for j in range(out_w):
            patch = x[i:i + kh, j:j + kw]   # (kh, kw)
            y[i, j] = np.sum(patch * k)     # 원소별 곱 후 합 (점곱)
    return y


# 5x5 입력에 3x3 커널 (가장자리 검출 비슷한 sobel-x 형태)
x = np.arange(25, dtype=np.float32).reshape(5, 5)
k = np.array([[-1, 0, 1],
              [-2, 0, 2],
              [-1, 0, 1]], dtype=np.float32)

print("입력:")
print(x)
print("커널 (Sobel-x):")
print(k)

y = conv2d_numpy(x, k)
print(f"출력 shape: {y.shape}  (예상: 5-3+1 = 3)")
print("출력:")
print(y)


# 2. Padding 과 Stride

기본 conv 는 출력이 입력보다 작아진다 ($H - k + 1$).
큰 네트워크에서는 conv 를 여러 번 쌓으면 feature map 이 너무 빨리 줄어들어 정보를 잃는다.

**Padding**: 입력 가장자리에 0 (또는 다른 값) 을 덧댄다.
- $p = (k - 1) / 2$ 로 두면 출력 크기 = 입력 크기 ("same" padding).
- 가장자리 픽셀이 한 번씩만 사용되는 것을 방지한다.

**Stride**: 커널이 한 번에 몇 칸 움직이는가.
- stride > 1 이면 다운샘플링 효과 (출력이 작아짐).

일반 출력 크기 공식:

$$ \text{out} = \left\lfloor \dfrac{N + 2p - k}{s} \right\rfloor + 1 $$

여기서 $N$ = 입력 한 변 길이, $p$ = padding, $k$ = 커널 크기, $s$ = stride.

In [ ]:
def out_size(N: int, k: int, p: int, s: int) -> int:
    """conv 출력 한 변 크기 공식."""
    return (N + 2 * p - k) // s + 1


# 다양한 옵션의 출력 크기 표
print(f"{'N':>3} {'k':>3} {'p':>3} {'s':>3} -> {'out':>4}")
for (N, k, p, s) in [
    (5, 3, 0, 1),   # 기본
    (5, 3, 1, 1),   # same padding
    (5, 3, 0, 2),   # stride 2 (다운샘플)
    (7, 3, 1, 2),   # padding + stride 2
    (28, 3, 1, 1),  # MNIST 한 장에 same padding
    (28, 5, 0, 1),  # 5x5 커널, no padding
]:
    print(f"{N:>3} {k:>3} {p:>3} {s:>3} -> {out_size(N, k, p, s):>4}")

# padding 효과를 numpy 로 직접 확인 — 같은 입력에 padding 만 추가
x = np.arange(25, dtype=np.float32).reshape(5, 5)
x_padded = np.pad(x, pad_width=1, mode='constant', constant_values=0)
print("\npadded 입력 shape:", x_padded.shape, "(5+2 = 7)")
print(x_padded)
y_same = conv2d_numpy(x_padded, k)
print(f"same padding 출력 shape: {y_same.shape}  (입력과 같아짐)")


# 3. PyTorch `nn.Conv2d` 와 비교

위에서 numpy 로 구현한 결과와 PyTorch 의 `nn.Conv2d` 결과가 정확히 일치하는지 확인한다.

PyTorch 의 입력 모양은 항상 **`(N, C, H, W)`**: batch, channel, height, width.
`nn.Conv2d` 의 가중치 모양은 **`(C_out, C_in, kh, kw)`**.

같은 커널 / 같은 입력을 넣으면 numpy 결과와 같아야 한다.

In [ ]:
# (a) numpy 결과
x_np = np.arange(25, dtype=np.float32).reshape(5, 5)
k_np = np.array([[-1, 0, 1],
                 [-2, 0, 2],
                 [-1, 0, 1]], dtype=np.float32)
y_np = conv2d_numpy(x_np, k_np)

# (b) PyTorch nn.Conv2d
# bias 를 끄고, 같은 커널을 직접 주입한다.
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3,
                 stride=1, padding=0, bias=False)
with torch.no_grad():
    # weight shape: (C_out=1, C_in=1, kh=3, kw=3)
    conv.weight.copy_(torch.from_numpy(k_np).reshape(1, 1, 3, 3))

x_t = torch.from_numpy(x_np).reshape(1, 1, 5, 5)   # (N=1, C=1, H=5, W=5)
y_t = conv(x_t).detach().numpy().reshape(3, 3)

print("numpy:")
print(y_np)
print("\nPyTorch:")
print(y_t)

# 핵심 검증
assert np.allclose(y_np, y_t, atol=1e-6), "numpy 와 PyTorch 결과가 다름!"
print("\nOK — 두 결과가 정확히 일치한다.")


# 4. 채널 (입력 채널, 출력 채널, 커널)

이미지 한 장은 보통 **여러 채널** 을 가진다 (RGB 면 3 채널).
또 한 conv 층은 보통 **여러 출력 채널** 을 만든다 (서로 다른 필터 여러 개).

따라서 `nn.Conv2d` 의 가중치 텐서 모양은:

$$ W \in \mathbb{R}^{C_\text{out} \times C_\text{in} \times k_h \times k_w} $$

각 출력 채널 $c$ 에 대해 **모든 입력 채널을 더해서** 한 장의 feature map 이 만들어진다:

$$ Y_{c,i,j} = \sum_{c'=0}^{C_\text{in}-1} \sum_{m,n} X_{c', i+m, j+n} \cdot W_{c, c', m, n} + b_c $$

**파라미터 수** (편향 포함):

$$ \#\text{params} = C_\text{out} \cdot C_\text{in} \cdot k_h \cdot k_w + C_\text{out} $$

In [ ]:
# 예: RGB (C_in=3) 이미지를 받아 16개 feature map 을 만드는 3x3 conv
C_in, C_out, kh, kw = 3, 16, 3, 3

conv = nn.Conv2d(in_channels=C_in, out_channels=C_out,
                 kernel_size=(kh, kw), bias=True)

# 직접 계산
expected = C_out * C_in * kh * kw + C_out
print(f"공식으로 계산한 파라미터 수: {C_out}*{C_in}*{kh}*{kw} + {C_out} = {expected}")

# PyTorch 가 자동으로 만든 파라미터 수
actual = sum(p.numel() for p in conv.parameters())
print(f"PyTorch 가 보고한 파라미터 수: {actual}")

assert expected == actual, "파라미터 수 불일치!"
print("OK — 일치.")

# weight / bias 모양도 확인
print(f"\nweight shape: {tuple(conv.weight.shape)}  (C_out, C_in, kh, kw)")
print(f"bias   shape: {tuple(conv.bias.shape)}  (C_out,)")


# 5. Pooling (Max / Avg)

Pooling 은 학습 가능한 파라미터가 **없는** 다운샘플링 연산이다.

- **MaxPool**: 윈도우 내에서 최댓값만 남긴다.
- **AvgPool**: 윈도우 내 평균을 낸다.

목적:
1. **다운샘플링** — feature map 크기를 줄여 연산량과 메모리 절감
2. **약한 translation invariance** — 입력이 한두 픽셀 움직여도 결과가 거의 같다

**Backward 동작 (직관)**:
- MaxPool 의 backward: 최댓값을 차지했던 위치로만 gradient 가 흐른다 (one-hot routing).
- AvgPool 의 backward: 윈도우 내 모든 위치에 gradient 를 균등하게 나눈다.

> 최근 SOTA 모델 (ResNet 이후) 은 pooling 대신 stride 가 큰 conv 로 다운샘플링하는 경우가 많지만,
> 여전히 입출력 모양 감각을 익히기엔 pooling 이 가장 명료하다.

In [ ]:
x = torch.tensor([[
    [[1.0, 3.0, 2.0, 4.0],
     [5.0, 6.0, 1.0, 2.0],
     [7.0, 8.0, 9.0, 0.0],
     [1.0, 2.0, 3.0, 4.0]]
]])  # shape: (N=1, C=1, H=4, W=4)

print("입력:")
print(x[0, 0])

# 2x2 max pool, stride 2 → 4x4 입력이 2x2 출력으로 줄어든다.
maxp = nn.MaxPool2d(kernel_size=2, stride=2)
avgp = nn.AvgPool2d(kernel_size=2, stride=2)

print("\nMaxPool2d(2, 2) 결과:")
print(maxp(x)[0, 0])

print("\nAvgPool2d(2, 2) 결과:")
print(avgp(x)[0, 0])

# 파라미터가 없는지 확인
print(f"\nMaxPool 파라미터 수: {sum(p.numel() for p in maxp.parameters())}")


# 6. 작은 CNN 빌드 — 모양을 추적하면서

전형적인 작은 CNN 의 흐름:

```
입력 (N, 1, 28, 28)            ← MNIST 한 장 (흑백)
 → Conv(1→8, k=3, p=1)         feature map 8장으로 늘림
 → ReLU
 → MaxPool(2,2)                14x14 로 다운샘플
 → Conv(8→16, k=3, p=1)        feature map 16장으로 늘림
 → ReLU
 → MaxPool(2,2)                7x7 로 다운샘플
 → Flatten                     16 * 7 * 7 = 784 차원 벡터
 → Linear(784 → 10)            클래스 점수
```

각 단계마다 모양이 어떻게 변하는지 직접 추적하는 습관이 매우 중요하다.

In [ ]:
class SmallCNN(nn.Module):
    """전형적인 작은 분류용 CNN."""

    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)   # (N,1,28,28) -> (N,8,28,28)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)  # (N,8,14,14) -> (N,16,14,14)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(16 * 7 * 7, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))     # (N,8,28,28)
        x = self.pool(x)              # (N,8,14,14)
        x = F.relu(self.conv2(x))     # (N,16,14,14)
        x = self.pool(x)              # (N,16,7,7)
        x = x.flatten(1)              # (N, 16*7*7)  ← batch 차원만 남기고 펼침
        x = self.fc(x)                # (N, n_classes)
        return x


# 모양 추적 — 더미 입력 한 번 흘려보내며 단계별 shape 출력
model = SmallCNN()
dummy = torch.randn(2, 1, 28, 28)   # batch=2

x = dummy
print(f"input          : {tuple(x.shape)}")
x = F.relu(model.conv1(x));  print(f"after conv1+ReLU: {tuple(x.shape)}")
x = model.pool(x);           print(f"after pool     : {tuple(x.shape)}")
x = F.relu(model.conv2(x));  print(f"after conv2+ReLU: {tuple(x.shape)}")
x = model.pool(x);           print(f"after pool     : {tuple(x.shape)}")
x = x.flatten(1);            print(f"after flatten  : {tuple(x.shape)}")
x = model.fc(x);             print(f"after fc       : {tuple(x.shape)}")


## 학습 — MNIST 또는 합성 데이터

`torchvision` 이 있으면 진짜 MNIST 로 1 epoch 학습한다.
없으면 28x28 합성 노이즈 + 랜덤 라벨로 같은 학습 루프가 동작하는지만 확인한다 (loss 가 줄지 않는 게 정상).

In [ ]:
import time

USE_MNIST = False
try:
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    transform = transforms.Compose([transforms.ToTensor()])
    # 데이터 다운로드는 시간이 걸릴 수 있다.
    train_ds = datasets.MNIST(root='./_mnist', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    USE_MNIST = True
    print(f"MNIST 사용: train 샘플 {len(train_ds)} 개")
except Exception as e:
    print(f"torchvision/MNIST 사용 불가 ({e!r}) — 합성 데이터로 대체")

if not USE_MNIST:
    # 합성 데이터: 노이즈 이미지 + 랜덤 라벨 (학습이 의미가 없으므로 loss 는 거의 안 줄어든다)
    from torch.utils.data import TensorDataset, DataLoader
    X = torch.randn(2048, 1, 28, 28)
    y = torch.randint(0, 10, (2048,))
    train_loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)

torch.manual_seed(SEED)
model = SmallCNN(n_classes=10)
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
loss_fn = nn.CrossEntropyLoss()

model.train()
t0 = time.time()
for step, (xb, yb) in enumerate(train_loader):
    opt.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    opt.step()
    if step % 50 == 0:
        acc = (logits.argmax(dim=1) == yb).float().mean().item()
        print(f"step {step:4d}  loss={loss.item():.4f}  batch_acc={acc:.3f}")
    if step >= 200:   # CPU 에서 너무 오래 걸리지 않게 200 step 만
        break
print(f"\n경과 시간: {time.time() - t0:.1f}s")


# 7. CNN 의 inductive bias — MLP 와 비교

CNN 이 이미지에서 잘 동작하는 본질적 이유는 **두 가지 사전지식 (inductive bias)** 때문이다.

1. **Locality (지역성)**
   - 이미지에서 의미 있는 패턴 (모서리, 코너, 작은 텍스처) 은 **가까운 픽셀끼리** 만들어진다.
   - 그래서 한 출력 픽셀을 만들 때 입력의 작은 patch (예: 3x3) 만 본다.
   - MLP 는 모든 입력 픽셀을 모든 출력 뉴런에 연결한다 — 위치 정보를 완전히 잃는다.

2. **Weight sharing (가중치 공유)**
   - 같은 커널이 **모든 위치** 에 적용된다.
   - "왼쪽 위에서 가장자리를 찾는 필터" 와 "오른쪽 아래에서 가장자리를 찾는 필터" 가 같다.
   - 결과: 파라미터 수가 극단적으로 적고, **위치가 약간 달라도 같은 패턴을 인식** (translation equivariance).

이 두 가정이 자연 이미지에 맞기 때문에 CNN 이 MLP 보다 훨씬 적은 데이터/파라미터로 잘 학습한다.

In [ ]:
# 같은 작업 (MNIST 28x28 -> 10 class) 을 MLP 로 만들었을 때 파라미터 수 비교
class MLP(nn.Module):
    def __init__(self, hidden: int = 128, n_classes: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


cnn = SmallCNN(n_classes=10)
mlp = MLP(hidden=128, n_classes=10)

cnn_params = sum(p.numel() for p in cnn.parameters())
mlp_params = sum(p.numel() for p in mlp.parameters())

print(f"SmallCNN 파라미터 수 : {cnn_params:>8,}")
print(f"MLP(128,128) 파라미터 수: {mlp_params:>8,}")
print(f"비율 (MLP / CNN)      : {mlp_params / cnn_params:.2f}x")

# 더 극적으로: hidden=512 짜리 MLP 와 비교
mlp_big = MLP(hidden=512)
print(f"\nMLP(512,512) 파라미터 수: {sum(p.numel() for p in mlp_big.parameters()):>8,}")
print("→ CNN 이 같은 task 를 훨씬 적은 파라미터로 표현 가능 (locality + weight sharing 의 힘)")


# 8. 흔한 함정

CNN 을 처음 쓸 때 가장 자주 막히는 지점들:

1. **입력 모양 (N, C, H, W) 순서를 잊음**
   - PyTorch 는 항상 channel-first.
   - `(28, 28)` 이미지 한 장을 그대로 conv 에 넣으면 에러 — `(1, 1, 28, 28)` 로 reshape 필요.

2. **Conv 출력 크기 계산 실수**
   - `Linear` 입력 차원을 손으로 적다가 틀린다.
   - 해결책: 더미 입력 한 번 흘려보고 출력 shape 을 출력해서 확인.

3. **BatchNorm 위치**
   - 보통 `Conv -> BN -> ReLU` 순서. ReLU 먼저 두면 음수 값이 0 이 된 뒤 정규화돼 의도와 어긋난다.

4. **channels_first vs channels_last**
   - matplotlib `imshow` 는 `(H, W, C)` 또는 `(H, W)` 를 기대.
   - PyTorch tensor 를 그대로 던지면 모양이 안 맞아서 깨져 보인다.

아래 셀에서 가장 자주 마주치는 두 가지를 시연한다.

In [ ]:
# 함정 1: 모양 순서를 잊고 conv 에 (H, W) 만 던짐
img = torch.randn(28, 28)   # (H, W) — 잘못된 모양
conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)

try:
    conv(img)
except RuntimeError as e:
    print("[함정 1: 모양 오류]")
    print(repr(e)[:200])

# 올바른 reshape: (N=1, C=1, H, W)
img_correct = img.unsqueeze(0).unsqueeze(0)   # (1, 1, 28, 28)
print(f"\n올바른 입력 shape: {tuple(img_correct.shape)}")
print(f"conv 출력 shape : {tuple(conv(img_correct).shape)}")

# 함정 2: Linear 입력 차원을 손으로 잘못 적음
class BadCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 16, kernel_size=3, padding=0)   # 28 -> 26
        self.pool = nn.MaxPool2d(2, 2)                            # 26 -> 13
        # WRONG: 14*14 라고 잘못 적음 (실제로는 13*13)
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv(x)))
        x = x.flatten(1)
        return self.fc(x)

try:
    BadCNN()(torch.randn(1, 1, 28, 28))
except RuntimeError as e:
    print("\n[함정 2: Linear 입력 차원 실수]")
    print(repr(e)[:200])
print("→ 더미 입력 한 번 흘려보면서 shape 을 직접 출력해 확인하는 습관이 최선의 예방.")


# 9. 참고 — 큰 사전학습 모델

처음부터 큰 CNN 을 짜는 일은 거의 없다.
**`torchvision.models`** 에 ResNet, VGG, EfficientNet 등 유명한 아키텍처가
사전학습 가중치까지 포함해 한 줄로 제공된다.

```python
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)   # 사전학습 가중치 다운로드
model.eval()
```

다음 단계로는:
- transfer learning (마지막 레이어만 새 task 에 맞게 교체)
- fine-tuning (전체 또는 일부 레이어를 작은 lr 로 추가 학습)
- data augmentation (`torchvision.transforms`)

가 일반적이다. 여기서는 모델을 한 줄로 만들어 파라미터 수만 확인한다.

In [ ]:
from torchvision.models import resnet18

model = resnet18(weights=None)   # 가중치 다운로드 안 함 (구조만 만들기)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet18 파라미터 수: {n_params:,}")
print(f"우리가 만든 SmallCNN: {sum(p.numel() for p in SmallCNN().parameters()):,}")
print()
print("ResNet18 의 첫 conv 층:")
print(model.conv1)
print("\nResNet18 의 마지막 fc 층:")
print(model.fc)


# 정리

| 개념              | 한 줄 요약                                                          |
|------------------|---------------------------------------------------------------------|
| Convolution      | sliding window 점곱 — locality + weight sharing                       |
| Padding/Stride   | 출력 크기를 제어 — `(N + 2p - k) / s + 1`                            |
| 채널             | 가중치 모양 `(C_out, C_in, kh, kw)`, 파라미터 = `C_out*C_in*kh*kw + C_out` |
| Pooling          | 학습 파라미터 없음, 다운샘플링 + 약한 translation invariance         |
| Inductive bias   | 자연 이미지의 locality 가정 → MLP 보다 훨씬 적은 파라미터로 학습     |
| 모양 디버깅      | 더미 입력 흘려보내고 step 별 `.shape` 출력하는 습관                  |
| 큰 모델          | `torchvision.models` 에 사전학습 가중치까지 포함                     |

다음 노트북 (`02_rnn_lstm_essentials.ipynb`) 에서는
**시간/시퀀스** 라는 또 다른 inductive bias 를 다루는 RNN/LSTM 을 본다.